In [1]:
import pandas as pd

In [2]:
# Loading a public dataset of country coordinates (name, latitude, longitude)
coords = pd.read_csv("https://raw.githubusercontent.com/gavinr/world-countries-centroids/master/dist/countries.csv")
coords.head()

,longitude,latitude,COUNTRY,ISO,COUNTRYAFF,AFF_ISO
0,-170.700732,-14.305712,American Samoa,AS,United States,US
1,166.638003,19.302046,United States Minor Outlying Islands,UM,United States,US
2,-159.787689,-21.222613,Cook Islands,CK,New Zealand,NZ
3,-149.400417,-17.674684,French Polynesia,PF,France,FR
4,-169.868781,-19.052309,Niue,NU,New Zealand,NZ


In [3]:
coords

,longitude,latitude,COUNTRY,ISO,COUNTRYAFF,AFF_ISO
0,-170.700732,-14.305712,American Samoa,AS,United States,US
1,166.638003,19.302046,United States Minor Outlying Islands,UM,United States,US
2,-159.787689,-21.222613,Cook Islands,CK,New Zealand,NZ
3,-149.400417,-17.674684,French Polynesia,PF,France,FR
4,-169.868781,-19.052309,Niue,NU,New Zealand,NZ
...,...,...,...,...,...,...
244,145.741197,15.178064,Northern Mariana Islands,MP,United States,US
245,134.579651,7.534776,Palau,PW,Palau,PW
246,98.670499,59.039434,Russian Federation,RU,Russian Federation,RU
247,-3.651625,40.365008,Spain,ES,Spain,ES


In [4]:
# Loading your own dataset's country names for comparison
data = pd.read_csv("../data/model_ready_data.csv")
your_countries = data["Country Name"].unique()

print("Number of countries in your dataset:", len(your_countries))
print(your_countries[:10])

Number of countries in your dataset: 227
<StringArray>
['Afghanistan, Islamic Rep. of',                      'Albania',
                      'Algeria',               'American Samoa',
     'Andorra, Principality of',                       'Angola',
                     'Anguilla',          'Antigua and Barbuda',
                    'Argentina',             'Armenia, Rep. of']
Length: 10, dtype: str


In [5]:
# Attempting a direct match first, to see how many countries line up as-is
coords_lookup = coords.set_index("COUNTRY")[["latitude", "longitude"]]

matched = [c for c in your_countries if c in coords_lookup.index]
unmatched = [c for c in your_countries if c not in coords_lookup.index]

print("Matched:", len(matched))
print("Unmatched:", len(unmatched))
print(unmatched[:20])

Matched: 157
Unmatched: 70
['Afghanistan, Islamic Rep. of', 'Andorra, Principality of', 'Armenia, Rep. of', 'Aruba, Kingdom of the Netherlands', 'Azerbaijan, Rep. of', 'Bahamas, The', 'Bahrain, Kingdom of', 'Belarus, Rep. of', 'Central African Rep.', 'China, P.R.: Hong Kong', 'China, P.R.: Macao', 'China, P.R.: Mainland', 'Comoros, Union of the', 'Congo, Dem. Rep. of the', 'Congo, Rep. of', 'Croatia, Rep. of', 'Czech Rep.', 'Dominican Rep.', 'Egypt, Arab Rep. of', 'Equatorial Guinea, Rep. of']


In [6]:
# Cleaning country names by removing everything after a comma, then trying to match again
import re

def clean_name(name):
    return re.split(r',', name)[0].strip()

# Building a lookup of original name -> cleaned name
name_mapping = {c: clean_name(c) for c in your_countries}

matched2 = {}
still_unmatched = []

for original, cleaned in name_mapping.items():
    if cleaned in coords_lookup.index:
        matched2[original] = cleaned
    else:
        still_unmatched.append(original)

print("Matched after cleaning:", len(matched2))
print("Still unmatched:", len(still_unmatched))
print(still_unmatched)

Matched after cleaning: 204
Still unmatched: 23
['Central African Rep.', 'Czech Rep.', 'Dominican Rep.', 'Falkland Islands (Malvinas)', 'Guiana, French', 'Holy See', "Korea, Dem. People's Rep. of", 'Korea, Rep. of', 'Kyrgyz Rep.', "Lao People's Dem. Rep.", 'Pitcairn Islands', 'Slovak Rep.', 'St. Kitts and Nevis', 'St. Lucia', 'St. Vincent and the Grenadines', 'Syrian Arab Rep.', 'São Tomé and Príncipe, Dem. Rep. of', 'Taiwan Province of China', 'United States Virgin Islands', 'Wallis and Futuna Islands', 'West Bank and Gaza', 'Western Sahara', 'World']


In [7]:
# Viewing all remaining unmatched countries
print(still_unmatched)

['Central African Rep.', 'Czech Rep.', 'Dominican Rep.', 'Falkland Islands (Malvinas)', 'Guiana, French', 'Holy See', "Korea, Dem. People's Rep. of", 'Korea, Rep. of', 'Kyrgyz Rep.', "Lao People's Dem. Rep.", 'Pitcairn Islands', 'Slovak Rep.', 'St. Kitts and Nevis', 'St. Lucia', 'St. Vincent and the Grenadines', 'Syrian Arab Rep.', 'São Tomé and Príncipe, Dem. Rep. of', 'Taiwan Province of China', 'United States Virgin Islands', 'Wallis and Futuna Islands', 'West Bank and Gaza', 'Western Sahara', 'World']


In [8]:
# Manual corrections for country names that don't follow the simple comma-stripping pattern
manual_fixes = {
    "Central African Rep.": "Central African Republic",
    "Czech Rep.": "Czech Republic",
    "Dominican Rep.": "Dominican Republic",
    "Falkland Islands (Malvinas)": "Falkland Islands",
    "Guiana, French": "French Guiana",
    "Korea, Dem. People's Rep. of": "North Korea",
    "Korea, Rep. of": "South Korea",
    "Kyrgyz Rep.": "Kyrgyzstan",
    "Lao People's Dem. Rep.": "Laos",
    "Pitcairn Islands": "Pitcairn",
    "Slovak Rep.": "Slovakia",
    "St. Kitts and Nevis": "Saint Kitts and Nevis",
    "St. Lucia": "Saint Lucia",
    "St. Vincent and the Grenadines": "Saint Vincent and the Grenadines",
    "Syrian Arab Rep.": "Syria",
    "São Tomé and Príncipe, Dem. Rep. of": "Sao Tome and Principe",
    "United States Virgin Islands": "US Virgin Islands",
    "Wallis and Futuna Islands": "Wallis and Futuna",
}

In [9]:
# Applying manual fixes on top of the cleaned names, then re-checking matches
final_mapping = {}
truly_unmatched = []

for original, cleaned in name_mapping.items():
    # Use manual fix if one exists for the original name, otherwise use the cleaned version
    lookup_name = manual_fixes.get(original, cleaned)

    if lookup_name in coords_lookup.index:
        final_mapping[original] = lookup_name
    else:
        truly_unmatched.append(original)

print("Final matched:", len(final_mapping))
print("Truly unmatched:", len(truly_unmatched))
print(truly_unmatched)

Final matched: 222
Truly unmatched: 5
['Holy See', 'Taiwan Province of China', 'West Bank and Gaza', 'Western Sahara', 'World']


In [10]:
# Building the final, complete country -> (latitude, longitude) lookup table
country_coordinates = {}

# Adding all successfully matched countries
for original, lookup_name in final_mapping.items():
    row = coords_lookup.loc[lookup_name]
    country_coordinates[original] = (row["latitude"], row["longitude"])

# Adding the manually handled edge cases
extra_coords = {
    "Holy See": (41.9029, 12.4534),
    "Taiwan Province of China": (23.6978, 120.9605),
    "West Bank and Gaza": (31.9522, 35.2332),
    "Western Sahara": (24.2155, -12.8858),
}
country_coordinates.update(extra_coords)

print("Total countries with coordinates:", len(country_coordinates))
print("Still missing:", set(your_countries) - set(country_coordinates.keys()))

Total countries with coordinates: 226
Still missing: {'World'}


In [11]:
data[data["Country Name"] == "World"]

,Country Name,year,temp_anomaly,lag_1,rolling_mean_5,decade,country_encoded
11373,World,1971,-0.089,0.153,0.0320,1970,223
11374,World,1972,-0.193,-0.089,-0.0430,1970,223
11375,World,1973,0.271,-0.193,0.0355,1970,223
11376,World,1974,-0.179,0.271,-0.0074,1970,223
11377,World,1975,0.091,-0.179,-0.0198,1970,223
11378,World,1976,-0.310,0.091,-0.0640,1970,223
11379,World,1977,0.273,-0.310,0.0292,1970,223
11380,World,1978,0.006,0.273,-0.0238,1970,223
11381,World,1979,0.227,0.006,0.0574,1970,223
11382,World,1980,0.335,0.227,0.1062,1980,223


In [12]:
import json

# Saving the country coordinates lookup so we never have to redo this matching again
with open("../data/country_coordinates.json", "w") as f:
    json.dump(country_coordinates, f)

print("Country coordinates saved.")

Country coordinates saved.


In [13]:
import json
with open("../data/country_coordinates.json") as f:
    check = json.load(f)

print(check["Nepal"])
print(len(check))

[28.300920699755657, 84.1338898313567]
226
